## University Salary Prediction

Given *data about university employees*, let's try to predict the **salary** of a given employee.

We will use a variety of regression models to make our predictions.

Data source: https://www.kaggle.com/datasets/tysonpo/university-salaries

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
data = pd.read_csv('archive/university-salaries/salaries_final.csv')
data

,Year,Name,Primary Job Title,Base Pay,Department,College
0,2010,"Abaied, Jamie L.",Assistant Professor,64000.0,Department of Psychological Science,CAS
1,2011,"Abaied, Jamie L.",Assistant Professor,64000.0,Department of Psychological Science,CAS
2,2012,"Abaied, Jamie L.",Assistant Professor,65229.0,Department of Psychological Science,CAS
3,2013,"Abaied, Jamie L.",Assistant Professor,66969.0,Department of Psychological Science,CAS
4,2014,"Abaied, Jamie L.",Assistant Professor,68658.0,Department of Psychological Science,CAS
...,...,...,...,...,...,...
14465,2016,"van der Vliet, Albert",Professor,163635.0,Department of Pathology&Laboratory Medicine,COM
14466,2017,"van der Vliet, Albert",Professor,175294.0,Department of Pathology&Laboratory Medicine,COM
14467,2018,"van der Vliet, Albert",Professor,191000.0,Department of Pathology&Laboratory Medicine,COM
14468,2019,"van der Vliet, Albert",Professor,196000.0,Department of Pathology&Laboratory Medicine,COM


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14470 entries, 0 to 14469
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               14470 non-null  int64  
 1   Name               14470 non-null  object 
 2   Primary Job Title  14470 non-null  object 
 3   Base Pay           14470 non-null  float64
 4   Department         14470 non-null  object 
 5   College            14470 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 678.4+ KB


### Preprocessing

In [4]:
df = data.copy()

In [5]:
# Dropping Name column
df = df.drop('Name', axis=1)

In [6]:
df

,Year,Primary Job Title,Base Pay,Department,College
0,2010,Assistant Professor,64000.0,Department of Psychological Science,CAS
1,2011,Assistant Professor,64000.0,Department of Psychological Science,CAS
2,2012,Assistant Professor,65229.0,Department of Psychological Science,CAS
3,2013,Assistant Professor,66969.0,Department of Psychological Science,CAS
4,2014,Assistant Professor,68658.0,Department of Psychological Science,CAS
...,...,...,...,...,...
14465,2016,Professor,163635.0,Department of Pathology&Laboratory Medicine,COM
14466,2017,Professor,175294.0,Department of Pathology&Laboratory Medicine,COM
14467,2018,Professor,191000.0,Department of Pathology&Laboratory Medicine,COM
14468,2019,Professor,196000.0,Department of Pathology&Laboratory Medicine,COM


In [7]:
# Shuffle the data
df = df.sample(frac=1.0).reset_index(drop=True)

In [8]:
# Split df into X and y
y = df['Base Pay']
X = df.drop('Base Pay', axis=1)

In [9]:
X

,Year,Primary Job Title,Department,College
0,2016,Research Professor,Department of Psychiatry,COM
1,2009,Research Associate Prof,Department of Med-Gen Internal Med,COM
2,2019,Assistant Professor,Department of Surg-Emergency Med,COM
3,2016,Professor,Department of Romance Languages,CAS
4,2017,Lecturer,Department of Music,CAS
...,...,...,...,...
14465,2019,Associate Professor,Department of Education,CESS
14466,2015,Associate Professor,Department of Biomedical and Health Sci,CNHS
14467,2020,Assistant Professor,Department of Surg-Ophthalmology,COM
14468,2016,Associate Professor,Department of Med-Gen Internal Med,COM


In [10]:
y

0         46659.0
1         70725.0
2         35000.0
3        100987.3
4         60142.0
           ...   
14465     68554.0
14466     92990.0
14467     35000.0
14468     24000.0
14469     97699.0
Name: Base Pay, Length: 14470, dtype: float64

### Building Pipeline

In [11]:
pd.get_dummies(X['College'], dtype=int)

,Business,CALS,CAS,CEMS,CESS,CNHS,COM,Department of Ext,LCOMEO,Learning and Info Tech,Library,RSENR
0,0,0,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
14465,0,0,0,0,1,0,0,0,0,0,0,0
14466,0,0,0,0,0,1,0,0,0,0,0,0
14467,0,0,0,0,0,0,1,0,0,0,0,0
14468,0,0,0,0,0,0,1,0,0,0,0,0


In [12]:
def build_pipeline(regressor):
    nominal_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('nominal', nominal_transformer, ['Primary Job Title', 'Department', 'College'])
    ], remainder='passthrough')

    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', StandardScaler()),
        ('regressor', regressor)
    ])

    return model

In [13]:
models = {
    'Linear Regression (Ridge)': build_pipeline(Ridge()),
    '            Decision Tree': build_pipeline(DecisionTreeRegressor()),
    '           Neural Network': build_pipeline(MLPRegressor()),
    '            Random Forest': build_pipeline(RandomForestRegressor()),
    '        Gradient Boosting': build_pipeline(GradientBoostingRegressor())
}

#### Model Selection (K-Fold Cross Validation)

In [14]:
def evaluate_model(model, X, y):
    kf = KFold(n_splits=5)
    rmses = []
    r2s = []
    
    for train_idx, test_idx in kf.split(X):
        # Fit the model
        model.fit(X.iloc[train_idx, :], y.iloc[train_idx])

        # Make predictions
        pred = model.predict(X.iloc[test_idx, :])

        # Calculate RMSE
        rmse = np.sqrt(np.mean((y.iloc[test_idx] - pred)**2))
        rmses.append(rmse)

        # Calculate R^2
        r2 = 1 - np.sum((y.iloc[test_idx] - pred)**2) / (np.sum((y.iloc[test_idx] - y.iloc[test_idx].mean())**2))
        r2s.append(r2)

    # Return average RMSE and R^2
    return np.mean(rmses), np.mean(r2s)

In [15]:
for name, model in models.items():
    print(name + " RMSE: {:.2f}".format(evaluate_model(model, X, y)[0]))

Linear Regression (Ridge) RMSE: 28529.38
            Decision Tree RMSE: 30183.01
           Neural Network RMSE: 31241.76
            Random Forest RMSE: 28899.44
        Gradient Boosting RMSE: 31628.12


In [16]:
for name, model in models.items():
    print(name + " R2: {:.4f}".format(evaluate_model(model, X, y)[1]))

Linear Regression (Ridge) R2: 0.6342
            Decision Tree R2: 0.5913
           Neural Network R2: 0.5649
            Random Forest R2: 0.6257
        Gradient Boosting R2: 0.5511
